Name: Rushikesh Anil Khairnar  
Student: 5115301  
Roll NO: 21  
College: B. K. Birla College

# **Practical 9**
**1.Build an end-to-end pipeline: CSV → Python/Pandas → Data Cleaning → SQL Database → Report.**



In [1]:
import pandas as pd
import sqlite3
import os

print("--- Task 1: End-to-End CSV to SQL Pipeline ---")

csv_file = 'raw_sales.csv'
db_file = 'task1_pipeline.db'

# 1. SETUP: Create messy raw data
with open(csv_file, 'w') as f:
    f.write("transaction_id,date,amount,category\n")
    f.write("1,2026-10-01,150.50,Electronics\n")
    f.write("2,,200.00,Clothing\n")          # Missing date
    f.write("3,2026-10-03,ERROR,Electronics\n") # Invalid amount string
    f.write("4,2026-10-04,99.99,Clothing\n")

# 2. EXTRACT: Read with Pandas
df = pd.read_csv(csv_file)

# 3. TRANSFORM & CLEAN
# Drop rows where 'date' is missing
df = df.dropna(subset=['date'])
# Convert 'amount' to numeric, forcing errors to NaN, then drop those NaNs
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df = df.dropna(subset=['amount'])
# Ensure proper date formatting
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

# 4. LOAD: Write to SQL Database
conn = sqlite3.connect(db_file)
df.to_sql('cleaned_sales', conn, if_exists='replace', index=False)

# 5. REPORT: Generate summary report using SQL
report_query = """
    SELECT category, COUNT(transaction_id) as total_sales, SUM(amount) as total_revenue
    FROM cleaned_sales
    GROUP BY category
"""
report_df = pd.read_sql(report_query, conn)

print("\nFinal Aggregated Report:")
print(report_df.to_string(index=False))

# Cleanup
conn.close()
os.remove(csv_file)
os.remove(db_file)

--- Task 1: End-to-End CSV to SQL Pipeline ---

Final Aggregated Report:
   category  total_sales  total_revenue
   Clothing            1          99.99
Electronics            1         150.50


**2.Build an ETL pipeline for an e-commerce dataset containing customers, orders, products, and payments.**


In [2]:
import pandas as pd
import sqlite3
import os

print("\n--- Task 2: E-Commerce ETL Pipeline ---")

db_file = 'task2_ecommerce.db'

# 1. EXTRACT: Simulated extraction of 4 different entities
customers = pd.DataFrame({
    'cust_id': [1, 2],
    'name': ['Alice', 'Bob']
})
products = pd.DataFrame({
    'prod_id': [101, 102],
    'price': [1200.0, 50.0]  # 101=Laptop, 102=Mouse
})
orders = pd.DataFrame({
    'order_id': [1001, 1002, 1003],
    'cust_id': [1, 1, 2],
    'prod_id': [101, 102, 102],
    'qty': [1, 2, 1]
})
payments = pd.DataFrame({
    'order_id': [1001, 1002, 1003],
    'status': ['Completed', 'Failed', 'Completed']
})

# 2. TRANSFORM
# Join Orders with Payments to filter only successful transactions
merged = orders.merge(payments, on='order_id')
successful_orders = merged[merged['status'] == 'Completed']

# Join with Products to calculate total transaction value
successful_orders = successful_orders.merge(products, on='prod_id')
successful_orders['total_value'] = successful_orders['qty'] * successful_orders['price']

# Join with Customers to get names and aggregate revenue per customer
final_merged = successful_orders.merge(customers, on='cust_id')
customer_revenue = final_merged.groupby('name')['total_value'].sum().reset_index()
customer_revenue.rename(columns={'total_value': 'lifetime_revenue'}, inplace=True)

# 3. LOAD
conn = sqlite3.connect(db_file)
customer_revenue.to_sql('customer_revenue_report', conn, if_exists='replace', index=False)

# 4. VISIBLE OUTPUT
print("\nFinal Customer Revenue Table (Notice Bob's failed order is excluded):")
print(pd.read_sql("SELECT * FROM customer_revenue_report", conn).to_string(index=False))

# Cleanup
conn.close()
os.remove(db_file)


--- Task 2: E-Commerce ETL Pipeline ---

Final Customer Revenue Table (Notice Bob's failed order is excluded):
 name  lifetime_revenue
Alice            1200.0
  Bob              50.0


**3.Design a data pipeline that handles both historical and newly arriving data.**

In [3]:
import pandas as pd
import sqlite3
import os

print("\n--- Task 3: Historical & Arriving Data Pipeline ---")

db_file = 'task3_incremental.db'
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# 1. SETUP: Create table and insert HISTORICAL data
cursor.execute('''CREATE TABLE user_profiles
                  (user_id INTEGER PRIMARY KEY, email TEXT, status TEXT)''')
historical_data = [
    (1, 'alice@test.com', 'Active'),
    (2, 'bob@test.com', 'Active')
]
cursor.executemany('INSERT INTO user_profiles VALUES (?, ?, ?)', historical_data)
conn.commit()

print("Initial Historical Data in DB:")
print(pd.read_sql("SELECT * FROM user_profiles", conn).to_string(index=False))

# 2. EXTRACT: Newly arriving data (Contains an update for ID 2, and a new record for ID 3)
new_data_df = pd.DataFrame({
    'user_id': [2, 3],
    'email': ['bob_updated@test.com', 'charlie@test.com'],
    'status': ['Inactive', 'Active']
})

# 3. TRANSFORM & LOAD: Incremental Upsert Strategy
# Step A: Load new data into a temporary staging table
new_data_df.to_sql('staging_profiles', conn, if_exists='replace', index=False)

# Step B: Perform UPSERT (Insert or Replace) from staging to main table based on Primary Key
cursor.execute('''
    INSERT OR REPLACE INTO user_profiles (user_id, email, status)
    SELECT user_id, email, status FROM staging_profiles
''')
conn.commit()

# Step C: Drop staging table
cursor.execute('DROP TABLE staging_profiles')
conn.commit()

# 4. VISIBLE OUTPUT
print("\nDatabase State After Incremental Load (ID 2 Updated, ID 3 Inserted):")
print(pd.read_sql("SELECT * FROM user_profiles", conn).to_string(index=False))

# Cleanup
conn.close()
os.remove(db_file)


--- Task 3: Historical & Arriving Data Pipeline ---
Initial Historical Data in DB:
 user_id          email status
       1 alice@test.com Active
       2   bob@test.com Active

Database State After Incremental Load (ID 2 Updated, ID 3 Inserted):
 user_id                email   status
       1       alice@test.com   Active
       2 bob_updated@test.com Inactive
       3     charlie@test.com   Active
